In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import property_management
import rent_payments

SOURCE_PATH = "/Volumes/property_catalog/property_management/landing_zone/rent_payments.csv"
TARGET_SCHEMA = "property_catalog.property_management"

RAW_TABLE = f"{TARGET_SCHEMA}.raw_rent_payments"
CLEAN_TABLE = f"{TARGET_SCHEMA}.clean_rent_payments"
REJECTED_TABLE = f"{TARGET_SCHEMA}.rejected_rent_payments"

In [0]:
rent_payments_df = property_management.extract_property_management_data(spark, SOURCE_PATH)

# Incrementally load raw rent payment records
property_management.upsert_property_management_table(spark, rent_payments_df, RAW_TABLE, key_column="Id")

display(rent_payments_df)

In [0]:
# Read raw rent payments Delta table
rent_payments_df = spark.table(RAW_TABLE)

transformed_rent_payments_df = rent_payments.transform_rent_payments(rent_payments_df)

In [0]:
validated_rent_payments_df = rent_payments.validate_rent_payments(transformed_rent_payments_df)

display(validated_rent_payments_df)

In [0]:
clean_rent_payments_df, rejected_rent_payments_df = rent_payments.separate_rent_payments(validated_rent_payments_df)

print("Clean rent payments:")
display(clean_rent_payments_df)

print("Rejected rent payments:")
display(rejected_rent_payments_df)

In [0]:
rent_payments_summary = property_management.summarise_property_management_etl(rent_payments_df, clean_rent_payments_df, rejected_rent_payments_df, SOURCE_PATH)

In [0]:
# Remove validation flags from the clean output
clean_rent_payments_output_df = clean_rent_payments_df.drop(
    "invalid_rent_payment_id",
    "duplicate_rent_payment_id",
    "invalid_amount",
    "invalid_payment_date",
    "invalid_status",
    "invalid_tenant_id"
)

# Incrementally load clean rent payment records
property_management.upsert_property_management_table(spark, clean_rent_payments_output_df, CLEAN_TABLE, key_column="Id")

# Incrementally load rejected rent payment records
property_management.upsert_property_management_table(spark, rejected_rent_payments_df, REJECTED_TABLE, key_column="Id")

print(f"Clean rent payments records saved to: {CLEAN_TABLE}")
print(f"Rejected rent payments records saved to: {REJECTED_TABLE}")